# 01 — TWOSIDES Dataset: Build Drug-Pair Labels + SMILES

**Pipeline:**
1. Load pre-computed `pair_stats.csv` (or stream from raw TWOSIDES if needed)
2. Apply PRR / frequency thresholds → binary labels
3. Balance classes
4. Fetch canonical SMILES from PubChem
5. Merge & save final `drug_pairs_labeled.csv`

In [6]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle as sk_shuffle
import requests, time, os, gc

# ── Paths ──────────────────────────────────────────────
DATA_DIR      = 'D:/DDI Project/data'
PAIR_STATS    = f'{DATA_DIR}/pair_stats.csv'
SMILES_CACHE  = f'{DATA_DIR}/smiles_cache.csv'
FINAL_OUTPUT  = f'{DATA_DIR}/drug_pairs_labeled.csv'

print('Imports OK')

Imports OK


## Step 1 — Load pair_stats

In [7]:
pair_stats = pd.read_csv(PAIR_STATS)
print(f'pair_stats loaded: {len(pair_stats):,} rows')
pair_stats.head(3)

pair_stats loaded: 211,989 rows


,drug_1_rxnorn_id,drug_1_concept_name,drug_2_rxnorm_id,drug_2_concept_name,max_PRR,max_freq,n_effects
0,10355,Temazepam,136411,sildenafil,50.0,0.128205,862
1,1808,Bumetanide,7824,Oxytocin,30.0,0.214286,118
2,221147,POLYETHYLENE GLYCOL 3350,5521,Hydroxychloroquine,40.0,0.100917,500


## Step 2 — Threshold analysis

In [8]:
print('Pairs meeting BOTH thresholds at different cutoffs:\n')
print(f'{"PRR":>6} {"Freq":>6} {"Positives":>12} {"Negatives":>12} {"Ratio":>8}')
print('-' * 50)

for prr_t in [5, 10, 20, 50]:
    for freq_t in [0.05, 0.10, 0.20]:
        pos = ((pair_stats['max_PRR']  >= prr_t) &
               (pair_stats['max_freq'] >= freq_t)).sum()
        neg = len(pair_stats) - pos
        ratio = pos / neg if neg > 0 else float('inf')
        print(f'{prr_t:>6} {freq_t:>6.2f} {pos:>12,} {neg:>12,} {ratio:>8.2f}')

Pairs meeting BOTH thresholds at different cutoffs:

   PRR   Freq    Positives    Negatives    Ratio
--------------------------------------------------
     5   0.05      211,811          178  1189.95
     5   0.10      187,977       24,012     7.83
     5   0.20       84,364      127,625     0.66
    10   0.05      211,740          249   850.36
    10   0.10      187,931       24,058     7.81
    10   0.20       84,347      127,642     0.66
    20   0.05      197,538       14,451    13.67
    20   0.10      176,167       35,822     4.92
    20   0.20       79,980      132,009     0.61
    50   0.05       81,699      130,290     0.63
    50   0.10       69,408      142,581     0.49
    50   0.20       30,870      181,119     0.17


## Step 3 — Label & balance

In [9]:
PRR_THRESHOLD  = 20.0
FREQ_THRESHOLD = 0.20

pair_stats['label'] = (
    (pair_stats['max_PRR']  >= PRR_THRESHOLD) &
    (pair_stats['max_freq'] >= FREQ_THRESHOLD)
).astype(int)

pos = pair_stats[pair_stats['label'] == 1]
neg = pair_stats[pair_stats['label'] == 0]

print(f'Positives available : {len(pos):,}')
print(f'Negatives available : {len(neg):,}')

n = min(len(pos), len(neg))
print(f'Using {n:,} from each class for balanced dataset')

pos_sample = pos.sample(n=n, random_state=42)
neg_sample = neg.sample(n=n, random_state=42)

df_labeled = pd.concat([pos_sample, neg_sample], ignore_index=True)
df_labeled = sk_shuffle(df_labeled, random_state=42).reset_index(drop=True)

print(f'\nFinal balanced dataset: {len(df_labeled):,} pairs')
print(f'Positives : {(df_labeled["label"]==1).sum():,}')
print(f'Negatives : {(df_labeled["label"]==0).sum():,}')

Positives available : 79,980
Negatives available : 132,009
Using 79,980 from each class for balanced dataset

Final balanced dataset: 159,960 pairs
Positives : 79,980
Negatives : 79,980


## Step 4 — Fetch SMILES from PubChem

**Bug fix:** The original code requested `CanonicalSMILES` from PubChem but the
API now returns the value under a *different* JSON key (e.g. `SMILES` or
`ConnectivitySMILES`). The `KeyError` was silently swallowed by the bare
`except: pass`, so **every** lookup returned `None`.

The fix below extracts the SMILES from whichever key PubChem provides.

In [10]:
# Delete the old corrupt cache (all values are NaN)
if os.path.exists(SMILES_CACHE):
    check = pd.read_csv(SMILES_CACHE)
    n_valid = check['smiles'].notna().sum()
    if n_valid == 0:
        os.remove(SMILES_CACHE)
        print(f'Deleted corrupt cache ({len(check)} entries, 0 valid SMILES)')
    else:
        print(f'Cache looks OK: {n_valid}/{len(check)} have SMILES')
else:
    print('No cache file found — will create from scratch')

Deleted corrupt cache (1898 entries, 0 valid SMILES)


In [11]:
# ── Collect unique drugs ──────────────────────────────
all_drugs = pd.concat([
    df_labeled[['drug_1_rxnorn_id','drug_1_concept_name']].rename(
        columns={'drug_1_rxnorn_id':'rxnorm_id',
                 'drug_1_concept_name':'name'}),
    df_labeled[['drug_2_rxnorm_id','drug_2_concept_name']].rename(
        columns={'drug_2_rxnorm_id':'rxnorm_id',
                 'drug_2_concept_name':'name'})
]).drop_duplicates(subset='rxnorm_id').reset_index(drop=True)

# Force IDs to clean strings
all_drugs['rxnorm_id'] = all_drugs['rxnorm_id'].astype(str).str.strip()

print(f'Unique drugs to look up: {len(all_drugs):,}')


# ── SMILES lookup function (FIXED) ────────────────────
def get_smiles(drug_name):
    """
    Query PubChem by drug name and return a canonical SMILES string.
    Handles the API returning the value under different JSON keys.
    """
    name_clean = str(drug_name).strip()
    url = (
        f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/'
        f'name/{requests.utils.quote(name_clean)}/'
        f'property/CanonicalSMILES/JSON'
    )
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            props = r.json()['PropertyTable']['Properties'][0]
            # PubChem may return the SMILES under different key names:
            #   'CanonicalSMILES', 'SMILES', 'ConnectivitySMILES', etc.
            for key in ('CanonicalSMILES', 'SMILES', 'ConnectivitySMILES',
                        'IsomericSMILES'):
                if key in props:
                    return props[key]
            # If none of the known keys, grab the first non-CID value
            for k, v in props.items():
                if k != 'CID' and isinstance(v, str):
                    return v
    except Exception as e:
        pass  # will return None
    return None


# ── Load cache ─────────────────────────────────────────
if os.path.exists(SMILES_CACHE):
    cache_df = pd.read_csv(SMILES_CACHE, dtype={'rxnorm_id': str})
    cache_df['rxnorm_id'] = cache_df['rxnorm_id'].str.strip()
    smiles_dict = {}
    for _, row in cache_df.iterrows():
        if pd.notna(row['smiles']):
            smiles_dict[row['rxnorm_id']] = row['smiles']
    print(f'Loaded {len(smiles_dict):,} valid cached entries')
else:
    smiles_dict = {}
    print('No cache — starting fresh')


# ── Look up missing drugs ──────────────────────────────
todo = all_drugs[~all_drugs['rxnorm_id'].isin(smiles_dict)].reset_index(drop=True)
print(f'Need to look up: {len(todo):,} drugs')
print(f'Estimated time : ~{len(todo) * 0.25 / 60:.1f} minutes')

for i, row in todo.iterrows():
    rxid = str(row['rxnorm_id'])
    smiles = get_smiles(row['name'])

    if smiles is not None:
        smiles_dict[rxid] = smiles

    # Progress + checkpoint every 100 drugs
    if (i + 1) % 100 == 0:
        # Save checkpoint
        pd.DataFrame(
            list(smiles_dict.items()),
            columns=['rxnorm_id', 'smiles']
        ).to_csv(SMILES_CACHE, index=False)
        pct = len(smiles_dict) / len(all_drugs) * 100
        print(f'  {i+1:>5}/{len(todo):>5} queried — '
              f'SMILES found so far: {len(smiles_dict):,} ({pct:.1f}%)')

    time.sleep(0.2)  # rate-limit: ~5 req/sec

# Final save
pd.DataFrame(
    list(smiles_dict.items()),
    columns=['rxnorm_id', 'smiles']
).to_csv(SMILES_CACHE, index=False)

found   = len(smiles_dict)
missing = len(all_drugs) - found
print(f'\nDone.')
print(f'SMILES found : {found:,} / {len(all_drugs):,} drugs')
print(f'Missing      : {missing:,}')

Unique drugs to look up: 1,898
No cache — starting fresh
Need to look up: 1,898 drugs
Estimated time : ~7.9 minutes
    100/ 1898 queried — SMILES found so far: 89 (4.7%)
    200/ 1898 queried — SMILES found so far: 180 (9.5%)
    300/ 1898 queried — SMILES found so far: 267 (14.1%)
    400/ 1898 queried — SMILES found so far: 359 (18.9%)
    500/ 1898 queried — SMILES found so far: 446 (23.5%)
    600/ 1898 queried — SMILES found so far: 534 (28.1%)
    700/ 1898 queried — SMILES found so far: 621 (32.7%)
    800/ 1898 queried — SMILES found so far: 712 (37.5%)
    900/ 1898 queried — SMILES found so far: 798 (42.0%)
   1000/ 1898 queried — SMILES found so far: 882 (46.5%)
   1100/ 1898 queried — SMILES found so far: 960 (50.6%)
   1200/ 1898 queried — SMILES found so far: 1,044 (55.0%)
   1300/ 1898 queried — SMILES found so far: 1,121 (59.1%)
   1400/ 1898 queried — SMILES found so far: 1,202 (63.3%)
   1500/ 1898 queried — SMILES found so far: 1,290 (68.0%)
   1600/ 1898 queried — 

## Step 5 — Merge SMILES & save final dataset

In [12]:
# Ensure ID columns are clean strings
df_labeled['drug_1_rxnorn_id'] = df_labeled['drug_1_rxnorn_id'].astype(str).str.strip()
df_labeled['drug_2_rxnorm_id'] = df_labeled['drug_2_rxnorm_id'].astype(str).str.strip()

# Map SMILES
df_labeled['Drug1_SMILES'] = df_labeled['drug_1_rxnorn_id'].map(smiles_dict)
df_labeled['Drug2_SMILES'] = df_labeled['drug_2_rxnorm_id'].map(smiles_dict)

# Check coverage
has_both = df_labeled['Drug1_SMILES'].notna() & df_labeled['Drug2_SMILES'].notna()
print(f'Pairs with BOTH SMILES : {has_both.sum():,} / {len(df_labeled):,}')
print(f'Drug1 missing SMILES   : {df_labeled["Drug1_SMILES"].isna().sum():,}')
print(f'Drug2 missing SMILES   : {df_labeled["Drug2_SMILES"].isna().sum():,}')

# Keep only rows with valid SMILES for both drugs
df_final = df_labeled[has_both][[
    'drug_1_concept_name', 'drug_2_concept_name',
    'Drug1_SMILES', 'Drug2_SMILES', 'label'
]].rename(columns={
    'drug_1_concept_name': 'Drug1_Name',
    'drug_2_concept_name': 'Drug2_Name'
}).reset_index(drop=True)

print(f'\nFinal dataset: {len(df_final):,} rows')
print(f'  Positives: {(df_final["label"]==1).sum():,}')
print(f'  Negatives: {(df_final["label"]==0).sum():,}')

# Save
df_final.to_csv(FINAL_OUTPUT, index=False)
print(f'\nSaved to {FINAL_OUTPUT}')

df_final.head()

Pairs with BOTH SMILES : 133,063 / 159,960
Drug1 missing SMILES   : 13,579
Drug2 missing SMILES   : 14,537

Final dataset: 133,063 rows
  Positives: 66,445
  Negatives: 66,618

Saved to D:/DDI Project/data/drug_pairs_labeled.csv


,Drug1_Name,Drug2_Name,Drug1_SMILES,Drug2_SMILES,label
0,pantoprazole,Nifedipine,COC1=C(C(=NC=C1)CS(=O)C2=NC3=C(N2)C=C(C=C3)OC(...,CC1=C(C(C(=C(N1)C)C(=O)OC)C2=CC=CC=C2[N+](=O)[...,0
1,Phenytoin,Primidone,C1=CC=C(C=C1)C2(C(=O)NC(=O)N2)C3=CC=CC=C3,CCC1(C(=O)NCNC1=O)C2=CC=CC=C2,1
2,Codeine,Carboplatin,CN1CCC23C4C1CC5=C2C(=C(C=C5)OC)OC3C(C=C4)O,C1CC(C1)(C(=O)[O-])C(=O)[O-].N.N.[Pt+2],0
3,Guaifenesin,ambrisentan,COC1=CC=CC=C1OCC(CO)O,CC1=CC(=NC(=N1)OC(C(=O)O)C(C2=CC=CC=C2)(C3=CC=...,1
4,temozolomide,Fluconazole,CN1C(=O)N2C=NC(=C2N=N1)C(=O)N,C1=CC(=C(C=C1F)F)C(CN2C=NC=N2)(CN3C=NC=N3)O,0


In [13]:
# Quick sanity check — verify a few SMILES
print('Sample entries:')
for _, row in df_final.sample(5, random_state=0).iterrows():
    print(f'  {row["Drug1_Name"]:>25s}  +  {row["Drug2_Name"]:<25s}  label={row["label"]}')
    print(f'     SMILES1: {row["Drug1_SMILES"][:60]}')
    print(f'     SMILES2: {row["Drug2_SMILES"][:60]}')
    print()

Sample entries:
                  Econazole  +  Lorazepam                  label=1
     SMILES1: C1=CC(=CC=C1COC(CN2C=CN=C2)C3=C(C=C(C=C3)Cl)Cl)Cl
     SMILES2: C1=CC=C(C(=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O)Cl

                 Prednisone  +  Methimazole                label=0
     SMILES1: CC12CC(=O)C3C(C1CCC2(C(=O)CO)O)CCC4=CC(=O)C=CC34C
     SMILES2: CN1C=CNC1=S

                raltegravir  +  Epoetin Alfa               label=1
     SMILES1: CC1=NN=C(O1)C(=O)NC(C)(C)C2=NC(=C(C(=O)N2C)O)C(=O)NCC3=CC=C(
     SMILES2: CC1CC=CC=CC(C(CC(C(C(C(CC(=O)O1)O)OC)OC2C(C(C(C(O2)C)OC3CC(C

                montelukast  +  hydroxyurea                label=0
     SMILES1: CC(C)(C1=CC=CC=C1CCC(C2=CC=CC(=C2)C=CC3=NC4=C(C=CC(=C4)Cl)C=
     SMILES2: C(=O)(N)NO

                Epinephrine  +  gatifloxacin               label=1
     SMILES1: CNCC(C1=CC(=C(C=C1)O)O)O
     SMILES2: CC1CN(CCN1)C2=C(C=C3C(=C2OC)N(C=C(C3=O)C(=O)O)C4CC4)F

